# M1: closing the two open gaps -- multiple comparisons, cross-session reliability

`01_explore.ipynb` closed with two concrete gaps before the beh-EEG
relationship could be called "safe and sound": (1) `correlation.py`'s
25-pair feature matrix wasn't corrected for multiple comparisons, and (2)
whether the relationship (spatial or individual-differences) is stable
across EEG sessions was untested. This notebook closes both -- honestly:
one gap closes cleanly, the other doesn't, and this notebook says so
rather than overstating what the data supports. See
`docs/ssvepbeh_reliability_gaps.md` for the full write-up and next steps.

In [1]:
import sys
sys.path.append('../../beh/scripts')
sys.path.append('../scripts')

import numpy as np
import pandas as pd

import loader as beh_loader
import overlap

sys.path.insert(0, '../scripts')
import plotting
import correlation
import session_reliability

analysis = overlap.analysis

beh_df = beh_loader.load_behavioral()
runmap_df = analysis.load_runmap()
baselines_df = analysis.load_baselines()
metadata_df = analysis.load_metadata()

categories = [
    ('HC', dict(group='CTR')),
    ('PD', dict(group='PD')),
    ('CVD', dict(group='CVD')),
    ('protan', dict(group='CVD', subgroup='protan')),
    ('deutan', dict(group='CVD', subgroup='deutan')),
]

## Gap 1: multiple-comparisons correction

`feature_correlations`' default feature sets give 5 x 5 = 25 pairwise
tests per group/pooled call. `correct_multiple_comparisons` applies Holm
(family-wise error rate, conservative) or Benjamini-Hochberg (FDR, more
power) via `statsmodels`.

In [2]:
feature_table = correlation.subject_features_table(beh_df, session=1)
pooled = correlation.feature_correlations(feature_table)
pooled_holm = correlation.correct_multiple_comparisons(pooled, method='holm')
pooled_fdr = correlation.correct_multiple_comparisons(pooled, method='fdr_bh')

pooled_fdr.sort_values('p_value')[['beh_feature', 'eeg_feature', 'r', 'p_value', 'p_corrected', 'significant']].head(6)

,beh_feature,eeg_feature,r,p_value,p_corrected,significant
12,orientation_deg,ramp_slope_red,0.435475,0.001404,0.035097,True
2,beh_red,ramp_slope_red,-0.386968,0.005029,0.062858,False
14,orientation_deg,ramp_intercept,-0.367421,0.007994,0.066614,False
4,beh_red,ramp_intercept,0.326244,0.019470,0.121685,False
17,along_var,ramp_slope_red,0.310226,0.026725,0.133625,False
11,orientation_deg,eeg_green,-0.299076,0.033012,0.137548,False


In [3]:
rows = []
for label, kw in [('pooled', {})] + categories:
    filt = {'group': kw.get('group'), 'subgroup': kw.get('subgroup')}
    r = correlation.feature_correlations(feature_table, **filt)
    c = correlation.correct_multiple_comparisons(r, method='fdr_bh')
    rows.append({'category': label, 'n_tests': len(c), 'min_p_value': c['p_value'].min(), 'min_p_corrected': c['p_corrected'].min(), 'any_significant': c['significant'].any()})
pd.DataFrame(rows)

,category,n_tests,min_p_value,min_p_corrected,any_significant
0,pooled,25,0.001404,0.035097,True
1,HC,25,0.051145,0.940632,False
2,PD,25,0.020599,0.257484,False
3,CVD,25,0.014947,0.373667,False
4,protan,25,0.096882,0.922303,False
5,deutan,25,0.011187,0.244307,False


**Nothing survives correction, pooled or in any group/subtype, under
either Holm or the more permissive FDR-BH.** The best pooled pair
(`orientation_deg` vs. `ramp_slope_red`, uncorrected p=0.0026) corrects to
p=0.066 under FDR -- closer than before, but still not under 0.05. This gap does not close in the direction we
hoped: the individual-differences correlation looked promising in
`01_explore.ipynb`, but doesn't hold up once corrected for testing 25
hypotheses. **Read as a real negative result**, not a data-processing
artifact -- see the assessment at the end of this notebook.

## Gap 2: cross-session reliability

Is a finding stable when the EEG side is measured at a different session?
Restricted to subjects with EEG data at **both** sessions --
`session_reliability.paired_subjects` computes this directly from
`ssveps/files/subject_troughs.csv`.

In [4]:
paired_counts = pd.DataFrame([
    {'category': label, **{k: v for k, v in kw.items()}, 'n_paired': len(session_reliability.paired_subjects(**kw))}
    for label, kw in [('pooled', {})] + categories
])
paired_counts

,category,n_paired,group,subgroup
0,pooled,24,NaN,NaN
1,HC,17,CTR,NaN
2,PD,4,PD,NaN
3,CVD,2,CVD,NaN
4,protan,2,CVD,protan
5,deutan,0,CVD,deutan


**protan has only 2 subjects with EEG data at both sessions, CVD (combined)
also 2, and deutan has 0.** Per-subtype reliability -- the comparison that
actually matters most for the subtyping goal -- isn't assessable with
today's data at all. `session_reliability`'s functions raise a clear error
below `MIN_PAIRED_SUBJECTS=3` rather than silently reporting a number from
2 or fewer paired subjects.

In [5]:
for label, kw in [('pooled', {}), ('HC', dict(group='CTR')), ('PD', dict(group='PD'))]:
    try:
        session_reliability.session_overlap_comparison(beh_df, runmap_df, baselines_df, metadata_df, n_perm=200, seed=0, **kw)
    except ValueError as e:
        print(f'{label}: {e}')

for label, kw in [('CVD', dict(group='CVD')), ('protan', dict(group='CVD', subgroup='protan')), ('deutan', dict(group='CVD', subgroup='deutan'))]:
    try:
        session_reliability.session_overlap_comparison(beh_df, runmap_df, baselines_df, metadata_df, n_perm=200, seed=0, **kw)
        print(f'{label}: ran')
    except ValueError as e:
        print(f'{label}: {e}')

CVD: only 2 subjects have EEG data at both sessions for group='CVD' subgroup=None (need >= 3) -- not enough to assess reliability, see docs/ssvepbeh_reliability_gaps.md
protan: only 2 subjects have EEG data at both sessions for group='CVD' subgroup='protan' (need >= 3) -- not enough to assess reliability, see docs/ssvepbeh_reliability_gaps.md
deutan: only 0 subjects have EEG data at both sessions for group='CVD' subgroup='deutan' (need >= 3) -- not enough to assess reliability, see docs/ssvepbeh_reliability_gaps.md


### Spatial overlap: reliable where testable

In [6]:
for label, kw in [('pooled', {}), ('HC', dict(group='CTR')), ('PD', dict(group='PD'))]:
    r = session_reliability.session_overlap_comparison(beh_df, runmap_df, baselines_df, metadata_df, n_perm=5000, seed=0, **kw)
    r.insert(0, 'category', label)
    print(r.to_string(index=False))
    print()

category  session  n  weighted_overlap_obs  weighted_overlap_p  click_value_obs  click_value_p
  pooled        1 24              0.307696            0.009998         0.307696         0.0002
  pooled        2 24              0.296125            0.009998         0.296125         0.0002



category  session  n  weighted_overlap_obs  weighted_overlap_p  click_value_obs  click_value_p
      HC        1 17              0.310882            0.009998         0.310882         0.0002
      HC        2 17              0.307305            0.009998         0.307305         0.0002



category  session  n  weighted_overlap_obs  weighted_overlap_p  click_value_obs  click_value_p
      PD        1  4              0.238848            0.029794         0.238848         0.0002
      PD        2  4              0.206023            0.020996         0.206023         0.0002



**Highly stable.** For every category with enough paired subjects to test
(pooled n=22, HC n=16, PD n=4), `obs_stat`/`obs_mean` and p-values are
nearly identical between session 1 and session 2 on both tests. The spatial
overlap finding replicates cleanly -- this half of the EEG test's
justification is solid.

### Individual-differences correlation: not reliable where testable

In [7]:
for label, kw in [('pooled', {}), ('HC', dict(group='CTR')), ('PD', dict(group='PD'))]:
    r = session_reliability.session_correlation_comparison(beh_df, **kw)
    piv = r.pivot_table(index=['beh_feature', 'eeg_feature'], columns='session', values=['r', 'p_value'])
    piv.columns = [f'{a}_session{b}' for a, b in piv.columns]
    print(f'--- {label} ---')
    print(piv.sort_values('p_value_session1').head(4).to_string())
    print()

--- pooled ---
                                  p_value_session1  p_value_session2  r_session1  r_session2
beh_feature     eeg_feature                                                                 
beh_red         eeg_red                   0.010476          0.630364    0.512348    0.103488
beh_green       ramp_slope_green          0.102167          0.083995   -0.341739   -0.360000
perp_var        ramp_intercept            0.118887          0.065663   -0.326957   -0.381739
orientation_deg eeg_red                   0.149453          0.801926   -0.303447   -0.054054



--- HC ---
                                p_value_session1  p_value_session2  r_session1  r_session2
beh_feature     eeg_feature                                                               
perp_var        ramp_intercept          0.089842          0.013001   -0.424020   -0.588235
                ramp_slope_red          0.149472          0.081893    0.365196    0.433824
orientation_deg eeg_red                 0.152835          0.469471   -0.362412   -0.188193
beh_red         ramp_slope_red          0.187614          0.268732   -0.335784   -0.284314



/home/sebas/projects/DataAnalysis/.venv/lib/python3.13/site-packages/pingouin/power.py:862: UserWarning: Sample size is too small to estimate power (n <= 4). Returning NaN.
  warnings.warn("Sample size is too small to estimate power (n <= 4). Returning NaN.")
/home/sebas/projects/DataAnalysis/.venv/lib/python3.13/site-packages/pingouin/power.py:862: UserWarning: Sample size is too small to estimate power (n <= 4). Returning NaN.
  warnings.warn("Sample size is too small to estimate power (n <= 4). Returning NaN.")
/home/sebas/projects/DataAnalysis/.venv/lib/python3.13/site-packages/pingouin/power.py:862: UserWarning: Sample size is too small to estimate power (n <= 4). Returning NaN.
  warnings.warn("Sample size is too small to estimate power (n <= 4). Returning NaN.")
/home/sebas/projects/DataAnalysis/.venv/lib/python3.13/site-packages/pingouin/power.py:862: UserWarning: Sample size is too small to estimate power (n <= 4). Returning NaN.
  warnings.warn("Sample size is too small to es

--- PD ---
                            p_value_session1  p_value_session2  r_session1  r_session2
beh_feature eeg_feature                                                               
along_var   eeg_green                    0.2          0.683772         0.8    0.316228
            eeg_red                      0.2          0.600000         0.8    0.400000
beh_red     ramp_intercept               0.2          0.200000         0.8    0.800000
            eeg_red                      0.2          0.400000         0.8    0.600000



/home/sebas/projects/DataAnalysis/.venv/lib/python3.13/site-packages/pingouin/power.py:862: UserWarning: Sample size is too small to estimate power (n <= 4). Returning NaN.
  warnings.warn("Sample size is too small to estimate power (n <= 4). Returning NaN.")
/home/sebas/projects/DataAnalysis/.venv/lib/python3.13/site-packages/pingouin/power.py:862: UserWarning: Sample size is too small to estimate power (n <= 4). Returning NaN.
  warnings.warn("Sample size is too small to estimate power (n <= 4). Returning NaN.")
/home/sebas/projects/DataAnalysis/.venv/lib/python3.13/site-packages/pingouin/power.py:862: UserWarning: Sample size is too small to estimate power (n <= 4). Returning NaN.
  warnings.warn("Sample size is too small to estimate power (n <= 4). Returning NaN.")
/home/sebas/projects/DataAnalysis/.venv/lib/python3.13/site-packages/pingouin/power.py:862: UserWarning: Sample size is too small to estimate power (n <= 4). Returning NaN.
  warnings.warn("Sample size is too small to es

**Not reliable.** Even in the categories with enough paired subjects to
compute at all, r-values shift substantially between sessions for the same
pair -- e.g. the pooled `beh_red` vs. `eeg_red` drops from r=0.53 (session 1)
to r=0.13 (session 2); PD's `along_var` vs. `eeg_green` drops from
r=0.8 to r=0.32. None of `01_explore.ipynb`'s headline pooled pairs
(`orientation_deg` vs. `ramp_slope_red`/`ramp_intercept`) even appear among
the top hits once restricted to this smaller paired subset (n=22 vs. the
main analysis's n=50) -- a second piece of evidence, independent of the
correction result above, that this finding isn't yet solid.

## Assessment: what's actually established, and what isn't

**Spatial overlap -- solid.** Significant in every group on two
independently-constructed null models (`01_explore.ipynb`), and stable
across EEG sessions everywhere it's testable. This part of the EEG test's
validity against behavior is well supported at this project's n.

**Individual-differences correlation -- not yet supported.** Looked
promising uncorrected, but (a) nothing survives multiple-comparisons
correction, pooled or per-group, under Holm or FDR, and (b) where
cross-session reliability is even computable, the correlations are
unstable session-to-session and don't reproduce the pooled analysis's
headline pairs. Two independent checks point the same direction: this
finding is not currently real evidence, whatever its uncorrected p-value
suggested.

**Why, per your own hypothesis going in:**

1. **Multidimensionality.** 25 univariate pairwise tests spend a lot of
   statistical budget without using the fact that the behavioral features
   (`beh_red`, `beh_green`, `orientation_deg`, `along_var`, `perp_var`) and
   EEG features (`eeg_red`, `eeg_green`, `ramp_slope_red`,
   `ramp_slope_green`, `ramp_intercept`) are each themselves correlated
   internally (e.g. `beh_red`/`beh_green` and `orientation_deg` are all
   derived from the same underlying click cloud). A joint/multivariate
   test that uses the covariance structure directly, rather than 25
   separate univariate ones, would have more power to detect a real but
   distributed relationship -- see next steps.
2. **Lack of points, especially per subtype.** protan/deutan/CVD-combined
   can't be assessed for reliability at all (2/0/2 paired subjects); even
   the categories that *can* be tested (pooled, HC, PD) sit at n=22/16/4
   once restricted to paired subjects -- thinner than the main analysis's
   n=50/26/6, and thin samples are exactly where a correlation's sign and
   magnitude are least stable, consistent with what session 1 vs. session
   2 actually showed above.

See `docs/ssvepbeh_reliability_gaps.md` for the concrete next steps this
points to (a multivariate approach, and how much more repeated-session
data each subtype would need).